# 07 · Direct Preference Optimization (DPO)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/post-training/07_dpo.ipynb) &nbsp; 📖 Concept deck: [DPO, KTO and rejection sampling](https://nilesh-salpe.github.io/visual-ai/post-training/07-dpo-and-friends/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** the DPO loss, the implicit reward it defines, and training on preference pairs; then a check that DPO recovers the same optimal policy RLHF targets, with no reward model and no RL loop.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def log_softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=axis, keepdims=True))

def log_sigmoid(z):
    return -np.logaddexp(0, -z)

K = 6
theta_ref = np.array([1.0, 0.5, 0.0, 0.0, -0.5, -1.0]); pi_ref = softmax(theta_ref)
r_true = np.array([0.0, 1.0, -0.5, 2.0, 0.5, 1.5])   # hidden reward behind the human labels
BETA = 0.5
def optimal_policy(beta):
    return softmax(np.log(pi_ref) + r_true / beta)
def kl(p, q):
    return float((p * (np.log(p) - np.log(q))).sum())
def make_prefs(n, seed=0):
    """Pairs sampled from the reference policy, labelled by Bradley-Terry on r_true."""
    rng = np.random.default_rng(seed)
    a, b = rng.choice(K, n, p=pi_ref), rng.choice(K, n, p=pi_ref)
    keep = a != b; a, b = a[keep], b[keep]
    a_wins = rng.random(len(a)) < 1 / (1 + np.exp(-(r_true[a] - r_true[b])))
    return np.where(a_wins, a, b), np.where(a_wins, b, a)
yw, yl = make_prefs(20000)

## Task 1: The DPO loss

Rafailov et al. (2023) show the RLHF optimum $\pi^\star \propto \pi_{ref}\,e^{r/\beta}$ can be inverted: $r(x,y) = \beta\log\frac{\pi^\star(y|x)}{\pi_{ref}(y|x)} + \beta\log Z(x)$. Plug this into the Bradley-Terry loss and $Z$ cancels:
$$\mathcal{L}_{DPO} = -\mathbb{E}\Big[\log\sigma\Big(\beta\log\frac{\pi_\theta(y_w)}{\pi_{ref}(y_w)} - \beta\log\frac{\pi_\theta(y_l)}{\pi_{ref}(y_l)}\Big)\Big]$$
Implement it from log-probs, plus `implicit_reward(logp, logp_ref, beta)` $= \beta(\log\pi_\theta - \log\pi_{ref})$.

In [ ]:
def implicit_reward(logp, logp_ref, beta):
    raise NotImplementedError

def dpo_loss(lp_w, lp_l, ref_w, ref_l, beta):
    # TODO: -mean log_sigmoid(beta * ((lp_w - ref_w) - (lp_l - ref_l)))
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_dpo_loss(f):
    z = np.zeros(3)
    assert np.isclose(f(z, z, z, z, 0.5), np.log(2)), "policy == reference -> loss log 2"
    lw, ll, rw, rl = np.log([.5, .2]), np.log([.1, .3]), np.log([.25, .25]), np.log([.25, .25])
    want = -log_sigmoid(0.5 * ((lw - rw) - (ll - rl))).mean()
    assert np.isclose(f(lw, ll, rw, rl, 0.5), want)
    ir = implicit_reward_solution if f.__name__.endswith('_solution') else implicit_reward
    assert np.allclose(ir(np.log([.5]), np.log([.25]), 2.0), [2 * np.log(2)])

check("dpo_loss", _test_dpo_loss, dpo_loss)

## Task 2: Train with DPO

Our policy is a softmax over K responses with logits θ (start at `theta_ref`). With $z = \beta[(\log\pi(y_w) - \log\pi_{ref}(y_w)) - (\log\pi(y_l) - \log\pi_{ref}(y_l))]$ and $\nabla_\theta\log\pi(a) = e_a - \pi$, the gradient of the loss for one pair is
$$\nabla_\theta\mathcal{L} = -(1 - \sigma(z))\,\beta\,(e_{y_w} - e_{y_l})$$
(the $-\pi$ terms cancel). Run full-batch gradient descent on all pairs and return θ and the loss history.

In [ ]:
def train_dpo(yw, yl, beta=BETA, steps=1500, lr=2.0):
    theta = theta_ref.copy(); hist = []
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_train_dpo(f):
    th, hist = f(yw, yl)
    assert hist[-1] < hist[0], "loss must decrease"
    p = softmax(th)
    assert kl(optimal_policy(BETA), p) < 0.02, f"DPO should recover pi* ∝ pi_ref*exp(r/beta); KL = {kl(optimal_policy(BETA), p):.4f}"
    ir = BETA * (np.log(p) - np.log(pi_ref))
    assert np.corrcoef(ir, r_true)[0, 1] > 0.98, "implicit rewards should match the true rewards up to a constant"

check("train_dpo", _test_train_dpo, train_dpo)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: The DPO loss
def implicit_reward_solution(logp, logp_ref, beta):
    return beta * (logp - logp_ref)

def dpo_loss_solution(lp_w, lp_l, ref_w, ref_l, beta):
    return -log_sigmoid(beta * ((lp_w - ref_w) - (lp_l - ref_l))).mean()

check("dpo_loss (reference)", _test_dpo_loss, dpo_loss_solution)

In [ ]:
# @title Solution for Task 2: Train with DPO
def train_dpo_solution(yw, yl, beta=BETA, steps=1500, lr=2.0):
    theta = theta_ref.copy(); hist = []; lref = np.log(pi_ref)
    for _ in range(steps):
        lp = log_softmax(theta)
        z = beta * ((lp[yw] - lref[yw]) - (lp[yl] - lref[yl]))
        hist.append(-log_sigmoid(z).mean())
        c = (1 - 1 / (1 + np.exp(-z))) * beta
        grad = -(np.bincount(yw, c, K) - np.bincount(yl, c, K)) / len(yw)
        theta = theta - lr * grad
    return theta, hist

check("train_dpo (reference)", _test_train_dpo, train_dpo_solution)

---
## Experiments
**Same target, two routes.** RLHF: fit a reward model, then run RL with a KL penalty (notebooks 04–05). DPO: one classification-style loss directly on the policy. With enough data both aim at $\pi^\star \propto \pi_{ref}e^{r/\beta}$. Below, the effect of β and of data size, and the implicit rewards DPO learns (equal to the true rewards up to a constant).

In [ ]:
for beta in [0.1, 0.5, 2.0]:
    th, _ = train_dpo_solution(yw, yl, beta=beta); p = softmax(th)
    print(f"beta={beta}: KL(pi||ref)={kl(p, pi_ref):.3f}  E[r_true]={p @ r_true:.3f}  KL(pi*||pi)={kl(optimal_policy(beta), p):.4f}")
for n in [200, 2000, 20000]:
    a, b = make_prefs(n, seed=5); th, _ = train_dpo_solution(a, b); p = softmax(th)
    print(f"{n:6d} pairs: KL(pi*||pi) = {kl(optimal_policy(BETA), p):.4f}")
th, _ = train_dpo_solution(yw, yl); ir = BETA * (log_softmax(th) - np.log(pi_ref))
print("\nimplicit reward − mean:", np.round(ir - ir.mean(), 2), "\ntrue reward − mean:    ", np.round(r_true - r_true.mean(), 2))

---
## Takeaways
* DPO turns RLHF's objective into a supervised loss on preference pairs: no reward model, no sampling, no critic. That is why it is popular (Llama 3's post-training uses SFT, rejection sampling and DPO).
* The policy *is* the reward model: $\beta\log\frac{\pi_\theta}{\pi_{ref}}$ is its implicit reward.
* Caveats: DPO learns only from the fixed pairs it is given (offline). Iterating, by collecting new pairs from the current model, closes much of the gap with online RL.